# Homework 6 — Decision trees and ensemble learning — ML Zoomcamp 2026 cohort

Predict `fuel_efficiency_mpg` (a **regression** task) on the 2026 car fuel-efficiency dataset, the same file as HW1 and HW2, with a decision tree, a random forest and XGBoost. Questions: `homework6_2026.md`.

**Submit at:** https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw06 · platform deadline **Mon 2 Nov**. Target: submit on the Mon 26 Oct 08:00 session (tracker), so the midterm starts Tue 27 Oct.

> **Run this notebook with scikit-learn 1.8 or older.** The official options were computed with scikit-learn 1.7.2. Random forests give different numbers on 1.9: Q2 no longer lands on an option and the 3-decimal ranking in Q3 changes. The setup cell prints a warning if your kernel has 1.9+; START_HERE has the one-line environment.

Scaffold only: question cells are empty.

## Setup and dataset

In [6]:
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

print('scikit-learn', sklearn.__version__, '| xgboost', xgb.__version__, '| pandas', pd.__version__)
if tuple(int(x) for x in sklearn.__version__.split('.')[:2]) >= (1, 9):
    print('WARNING: scikit-learn 1.9+ gives different random-forest numbers than the official options '
          '(computed with 1.7.2). Switch to the zc-hw06 kernel, see START_HERE.')

df_raw = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df_raw.copy()
df.head()

scikit-learn 1.7.2 | xgboost 3.4.2 | pandas 3.1.0


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Data preparation

Check for missing values, then fill **all** of them with zeros.

> `RandomForestRegressor` refuses NaN (*"Input X contains NaN"*), so Q2 fails if a gap survives.

In [7]:
# 1. Count the gaps in each column
print(df.isnull().sum())

# 2. Fill every gap with 0
df = df.fillna(0)

# 3. Check: this must print 0
print('NaN left:', df.isnull().sum().sum())

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64
NaN left: 0


## Split — 60/20/20

The homework pins these exact calls (`random_state=1`). Then pull out `y_train`, `y_val` (and `y_test`) and remove `fuel_efficiency_mpg` from the dataframes **before** `to_dict`. Turn the dataframes into matrices with `DictVectorizer(sparse=True)`: fit on train, transform validation.

> Left in the dataframe, the target becomes a feature: Q1's tree then splits on `fuel_efficiency_mpg` itself.

In [8]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [9]:
# 0. Clean row numbers
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# 1. Save the answers (targets)
y_train = df_train.fuel_efficiency_mpg.values
y_val   = df_val.fuel_efficiency_mpg.values
y_test  = df_test.fuel_efficiency_mpg.values

# 2. Remove the answer column
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

# 3. One-hot encoding
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val   = dv.transform(df_val.to_dict(orient='records'))

print(len(df_train), len(df_val), len(df_test))
print(X_train.shape, X_val.shape)

6000 2000 2000
(6000, 16) (2000, 16)


## Q1. Decision tree with `max_depth=1` — which feature splits?

```python
DecisionTreeRegressor(max_depth=1)
```

Print the rule with `export_text(dt, feature_names=list(dv.get_feature_names_out()))`. Answer with the **original** column: a one-hot name such as `origin=USA` maps back to `origin`.

*(vehicle_weight / model_year / origin / fuel_type)*

In [10]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Q2. Random forest — validation RMSE

```python
RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
```

RMSE on validation: `root_mean_squared_error(y_val, y_pred)`.

> `mean_squared_error(..., squared=False)` from older tutorials was removed in scikit-learn 1.6 (`TypeError`). Use `root_mean_squared_error`, or `np.sqrt(mean_squared_error(...))`.

*(0.1837 / 1.837 / 18.37 / 183.7)*

In [11]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
print(f'RMSE: {rmse:.4f}')

RMSE: 1.8371


## Q3. `n_estimators` — which value gives the lowest validation RMSE?

`n_estimators` in `[10, 50, 100, 150]`, `random_state=1`. **Round each RMSE to 3 decimals, then compare.**

> This is the question the scikit-learn version flips (see the header). Check the setup cell's version line before you trust the ranking.

*(10 / 50 / 100 / 150)*

In [12]:
scores = []

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    scores.append((n, round(rmse, 3)))

df_scores = pd.DataFrame(scores, columns=['n_estimators', 'rmse'])
df_scores


,n_estimators,rmse
0,10,1.837
1,50,1.772
2,100,1.768
3,150,1.769


## Q4. Best `max_depth` by mean RMSE

`max_depth` in `[10, 15, 20, 25]`; for each, `n_estimators` in `[10, 50, 100, 150]`, `random_state=1`. Average the four RMSEs per depth; the lowest mean wins.

> 16 forests: about a minute. Store `(max_depth, n_estimators, rmse)` tuples and use `groupby('max_depth').rmse.mean()`, as in the lecture's 6.6 tables.

*(10 / 15 / 20 / 25)*

In [14]:
scores = []

for d in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d,
                                   random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        scores.append((d, n, rmse))

df_scores = pd.DataFrame(scores, columns=['max_depth', 'n_estimators', 'rmse'])
df_scores.groupby('max_depth').rmse.mean().round(4)


max_depth
10    1.7554
15    1.7840
20    1.7882
25    1.7863
Name: rmse, dtype: float64

In [15]:
df_scores.groupby('max_depth').rmse.mean().round(4).sort_values()

max_depth
10    1.7554
15    1.7840
25    1.7863
20    1.7882
Name: rmse, dtype: float64

## Q5. Most important feature

```python
RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
```

Pair `rf.feature_importances_` with `dv.get_feature_names_out()` (same order) and rank **only the four listed** features.

> A feature outside the list, or a one-hot column, can rank higher than all four. That does not change the question.

*(vehicle_weight / horsepower / acceleration / engine_displacement)*

## Q6. XGBoost — `eta` 0.3 vs 0.1

DMatrix for train and validation, a watchlist, 100 rounds with exactly these parameters, then the same with `'eta': 0.1`:

```python
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}
```

> `feature_names=list(dv.get_feature_names_out())`: XGBoost 3 rejects the bare NumPy array with a `TypeError`.
> With `reg:squarederror` the watchlist reports `val-rmse` by default; no `eval_metric` needed. Read the curve from `evals_result={}` (as in `../practice/06-trees_commented.ipynb`, 6.7) rather than parsing printed text.
> Compare the RMSE of the trained models (last round, or `predict` + `root_mean_squared_error`: the same number). Also look at the best round of each curve: does it tell the same story?

*(0.3 / 0.1 / both give equal value)*